# Rebi — bilgi tabanı ingest (Google Colab)

PDF’leri **Gemini** (`gemini-embedding-001`, backend ile aynı) ile vektörleyip **Supabase Postgres** `knowledge_*` tablolarına yazar.

**Akış (git gerekmez)**

1. **Runtime →** CPU yeter.
2. **🔑 Secrets:** `GEMINI_API_KEY`, `SUPABASE_DATABASE_URL` veya `DATABASE_URL`. İsteğe bağlı: `KNOWLEDGE_CATALOG_USER_ID` (yoksa sabit demo UUID kullanılır).
3. Aşağıda **repo ZIP** yükleyin: GitHub’da **Code → Download ZIP**.
4. **PDF** dosyaları veya `pdf.zip` yükleyin.
5. Hücreleri sırayla çalıştırın.

Bu not defteri **Colab dışında** (ör. Cursor) açılırsa `google.colab` olmaz; o zaman ortam değişkeni `REBI_REPO_ROOT` (repo kökü) ve `PDF_INGEST_DIR` (PDF klasörü) verin.


In [ ]:
# @title Kurulum
%pip install -q psycopg[binary] PyPDF2 google-genai python-dotenv


In [ ]:
# @title Repo: ZIP yükle (Colab) veya REBI_REPO_ROOT (yerel)
from pathlib import Path
import os, shutil, zipfile

WORK = Path("/content/rebi_notebook_work")
WORK.mkdir(parents=True, exist_ok=True)
UNZIP = WORK / "repo"
if UNZIP.exists():
    shutil.rmtree(UNZIP)
UNZIP.mkdir(parents=True)


def find_backend(search_root: Path) -> Path:
    search_root = Path(search_root)
    for ingest in search_root.rglob("ingest.py"):
        if ingest.parent.name != "knowledge":
            continue
        backend = ingest.parent.parent
        if (backend / "knowledge" / "ingest.py").exists():
            return backend.resolve()
    raise FileNotFoundError(f"backend/knowledge/ingest.py bulunamadı: {search_root}")


try:
    from google.colab import files

    print("GitHub → Code → Download ZIP — repo arşivini yükleyin:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("ZIP yüklenmedi")
    zname = next(iter(uploaded))
    zpath = WORK / zname
    zpath.write_bytes(uploaded[zname])
    if not str(zname).lower().endswith(".zip"):
        raise RuntimeError("Lütfen .zip dosyası yükleyin")
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(UNZIP)
    BACKEND_ROOT = find_backend(UNZIP)
except ImportError:
    root = os.environ.get("REBI_REPO_ROOT", "").strip()
    if not root:
        raise SystemExit(
            "Colab dışı: export REBI_REPO_ROOT=/tam/yol/rebi  (içinde backend/knowledge olmalı)"
        )
    rr = Path(root).resolve()
    if (rr / "backend" / "knowledge" / "ingest.py").exists():
        BACKEND_ROOT = (rr / "backend").resolve()
    elif (rr / "knowledge" / "ingest.py").exists() and rr.name == "backend":
        BACKEND_ROOT = rr
    else:
        BACKEND_ROOT = find_backend(rr)

os.chdir(BACKEND_ROOT)
print("BACKEND_ROOT:", BACKEND_ROOT)


In [ ]:
# @title Ortam (Colab Secrets veya yerel env)
import os


def _secret(name, default=None):
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return os.environ.get(name, default)


g = _secret("GEMINI_API_KEY")
if g:
    os.environ["GEMINI_API_KEY"] = g
db = _secret("SUPABASE_DATABASE_URL") or _secret("DATABASE_URL")
if db:
    os.environ["SUPABASE_DATABASE_URL"] = db
d2 = _secret("DATABASE_URL")
if d2:
    os.environ["DATABASE_URL"] = d2
uid = _secret("KNOWLEDGE_CATALOG_USER_ID")
if uid:
    os.environ["KNOWLEDGE_CATALOG_USER_ID"] = uid
os.environ.setdefault("KNOWLEDGE_CATALOG_USER_ID", "00000000-0000-4000-8000-000000000001")

print("GEMINI:", "ok" if os.environ.get("GEMINI_API_KEY") else "EKSİK")
print(
    "DB URI:",
    "ok" if (os.environ.get("SUPABASE_DATABASE_URL") or os.environ.get("DATABASE_URL")) else "EKSİK",
)


In [ ]:
# @title PDF klasörü: yükle (Colab) veya PDF_INGEST_DIR (yerel)
from pathlib import Path
import os, shutil, zipfile

PDF_DIR = Path("/content/rebi_pdfs")
if PDF_DIR.exists():
    shutil.rmtree(PDF_DIR)
PDF_DIR.mkdir(parents=True)

try:
    from google.colab import files

    print("PDF’ler veya pdf.zip yükleyin:")
    up = files.upload()
    for name, data in up.items():
        p = PDF_DIR / name
        p.write_bytes(data)
        if str(name).lower().endswith(".zip"):
            with zipfile.ZipFile(p) as zf:
                zf.extractall(PDF_DIR)
            p.unlink(missing_ok=True)
except ImportError:
    alt = os.environ.get("PDF_INGEST_DIR", "").strip()
    if not alt or not Path(alt).is_dir():
        raise SystemExit("Colab dışı: export PDF_INGEST_DIR=/pdf/klasörü")
    PDF_DIR = Path(alt).resolve()

pdfs = list(PDF_DIR.rglob("*.pdf"))
if not pdfs:
    raise SystemExit(f"Bu klasörde PDF yok: {PDF_DIR}")
PDF_DIR_STR = str(PDF_DIR.resolve())
print("PDF sayısı:", len(pdfs), "| PDF_DIR:", PDF_DIR_STR)


In [ ]:
# @title Ingest + embedding (gemini-embedding-001)
import subprocess, os, sys
from pathlib import Path

backend = Path(os.getcwd()).resolve()
if not (backend / "knowledge" / "ingest.py").exists():
    raise SystemExit(f"cwd backend değil: {backend} — önce repo ZIP hücresini çalıştırın")

uid = os.environ.get("KNOWLEDGE_CATALOG_USER_ID", "00000000-0000-4000-8000-000000000001")
cmd = [
    sys.executable,
    "-m",
    "knowledge.ingest",
    "--user",
    uid,
    "--folder",
    "data-pdfs",
    "--title",
    "Rebi Colab katalog",
    "--dir",
    PDF_DIR_STR,
    "--store-raw",
]
# Aynı dosyayı baştan vektörlemek için: cmd.append("--replace-existing")
print(" ".join(cmd))
r = subprocess.run(cmd, cwd=str(backend))
raise SystemExit(r.returncode)


**Sonrası (isteğe bağlı)** chunk sınıflandırma — `BACKEND_ROOT` ve kullanıcı UUID ile:

`python3 -m knowledge.classify_chunks --user <UUID> --folder data-pdfs --limit 200`

(Çalışma dizini: `backend` klasörü.)

Büyük kitaplarda ingest uzun sürebilir; zaman aşımında **Runtime → Run after** veya daha az PDF ile tekrar deneyin.
